# GPU Matrix Multiplication Benchmark

## Setup: check the GPU and libraries


In [35]:
!nvidia-smi

Wed Oct  7 23:49:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             26W /   70W |     839MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [36]:
import numpy as np
import cupy as cp
from numba import cuda

print("NumPy", np.__version__, "| CuPy", cp.__version__)
print("Numba sees a GPU:", cuda.is_available())
print("GPU:", cuda.get_current_device().name)

NumPy 2.1.3 | CuPy 14.0.1
Numba sees a GPU: True
GPU: Tesla T4


In [37]:
import time

rng = np.random.default_rng(seed=0)

def make_matrices(n):
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    return A, B

## Section 1: Pure Python Triple Loop

In [38]:
#matrix multiplication (the 3 loops show how this becomes n^3)
def matmul_python(A, B):
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            total = 0.0
            for k in range(n):
                total += A[i][k] * B[k][j]
            C[i][j] = total
    return C

In [39]:
A, B = make_matrices(64)
C_python = matmul_python(A.tolist(), B.tolist())
C_numpy = A @ B
print("Matches NumPy:", np.allclose(C_python, C_numpy, rtol=1e-4))

Matches NumPy: True


In [40]:
#doubling n should take about 8x longer as the loop is O(n^3)
for n in [32, 64, 128, 256]:
    A, B = make_matrices(n)
    A_list, B_list = A.tolist(), B.tolist()
    start = time.perf_counter()
    matmul_python(A_list, B_list)
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.3f} seconds")


n =   32: 0.003 seconds
n =   64: 0.022 seconds
n =  128: 0.134 seconds
n =  256: 1.216 seconds


## Section 2: NumPy (CPU, optimized BLAS)

In [41]:
#NumPy (CPU, optimized BLAS)
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    start = time.perf_counter()
    A @ B
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000039 seconds
n =   64: 0.000368 seconds
n =  128: 0.008029 seconds
n =  256: 0.007524 seconds
n =  512: 0.012320 seconds
n = 1024: 0.020757 seconds
n = 2048: 0.147266 seconds
n = 4096: 1.073194 seconds


## Stage 3: CuPy (NVIDIA GPU via cuBLAS)

In [42]:
# comparing the speed of NumPy and python
A, B = make_matrices(256)
A_list, B_list = A.tolist(), B.tolist()

start = time.perf_counter()
matmul_python(A_list, B_list)
python_time = time.perf_counter() - start

start = time.perf_counter()
A @ B
numpy_time = time.perf_counter() - start

print(f"Python: {python_time:.3f} s | NumPy: {numpy_time:.6f} s | Speedup: {python_time / numpy_time:,.0f}x")

Python: 1.240 s | NumPy: 0.000676 s | Speedup: 1,835x


In [43]:
# using CuPy instead by copying matrices to GPU memory to multiply,
# then copying the result back to check it
A, B = make_matrices(512)
A_gpu = cp.asarray(A)
B_gpu = cp.asarray(B)
C_gpu = A_gpu @ B_gpu
print("Matches NumPy:", np.allclose(cp.asnumpy(C_gpu), A @ B, rtol=1e-4))

Matches NumPy: True


In [44]:
# GPU calls return immediately, so timing without synchronize is wrong
A, B = make_matrices(4096)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
A_gpu @ B_gpu
cp.cuda.Device().synchronize()

start = time.perf_counter()
A_gpu @ B_gpu
no_sync = time.perf_counter() - start

start = time.perf_counter()
A_gpu @ B_gpu
cp.cuda.Device().synchronize()
with_sync = time.perf_counter() - start

print(f"Without synchronize: {no_sync:.6f} s  wrong b/c only measures sending the job")
print(f"With synchronize:    {with_sync:.6f} s  correct timing")

Without synchronize: 0.000260 s  wrong b/c only measures sending the job
With synchronize:    0.095071 s  correct timing


In [45]:
# The first GPU call includes one-time setup, so it's much slower than later calls
A, B = make_matrices(1024)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
for attempt in range(1, 4):
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    print(f"Call {attempt}: {time.perf_counter() - start:.6f} s")

Call 1: 0.000625 s
Call 2: 0.000601 s
Call 3: 0.000526 s


In [46]:
# GPU compute time only (data already on the GPU), with warm-up and synchronize
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
    A_gpu @ B_gpu                      # warm-up for this size
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000099 seconds
n =   64: 0.000078 seconds
n =  128: 0.000047 seconds
n =  256: 0.000067 seconds
n =  512: 0.000152 seconds
n = 1024: 0.000493 seconds
n = 2048: 0.003256 seconds
n = 4096: 0.026784 seconds


## Stage 4: Custom CUDA kernel (Numba)

In [47]:
#custom CUDA kernel (Numba)
from numba import float32

#each GPU thread computes an entry of C with its location stored
#this is why we don't need the i and j loops from the python example
#x = column, so the 32 threads in a warp read neighboring memory (coalesced)
@cuda.jit
def matmul_kernel(A, B, C):
    col, row = cuda.grid(2)               # this thread's position in the grid
    n = C.shape[0]
    if row < n and col < n:               # skip threads that fall off the edge
        total = float32(0.0)
        for k in range(n):
            total += A[row, k] * B[k, col]
        C[row, col] = total

In [48]:
#copy inputs to the GPU, creating an empty output, and launch one thread per entry of C
n = 512
A, B = make_matrices(n)
A_dev = cuda.to_device(A)
B_dev = cuda.to_device(B)
C_dev = cuda.device_array((n, n), dtype=np.float32)

threads_per_block = (16, 16)
blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
cuda.synchronize()

print("Matches NumPy:", np.allclose(C_dev.copy_to_host(), A @ B, rtol=1e-4))

Matches NumPy: True


In [49]:
import warnings
warnings.filterwarnings("ignore", message="Grid size")   # small grids underuse the GPU; expected here

#kernel compute time only, with warm-up
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array((n, n), dtype=np.float32)
    blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

    matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)   # warm-up
    cuda.synchronize()

    start = time.perf_counter()
    matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
    cuda.synchronize()
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000092 seconds
n =   64: 0.000133 seconds
n =  128: 0.000111 seconds
n =  256: 0.000246 seconds
n =  512: 0.001239 seconds
n = 1024: 0.009249 seconds
n = 2048: 0.079314 seconds
n = 4096: 0.627624 seconds


### Testing thread mapping and memory coalescing

Same math, but seeing how threads handle consecutive rows instead of columns.

In [50]:
# Slower version for comparison: x = row, so a warp reads down a column of A (scattered in memory)
@cuda.jit
def matmul_kernel_uncoalesced(A, B, C):
    row, col = cuda.grid(2)
    n = C.shape[0]
    if row < n and col < n:
        total = float32(0.0)
        for k in range(n):
            total += A[row, k] * B[k, col]
        C[row, col] = total

In [51]:
# same math, different thread-to-entry mapping
for n in [2048, 4096]:
    A, B = make_matrices(n)
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array((n, n), dtype=np.float32)
    blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

    for name, kernel in [("uncoalesced", matmul_kernel_uncoalesced), ("coalesced", matmul_kernel)]:
        kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)   # warm-up
        cuda.synchronize()
        start = time.perf_counter()
        kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
        cuda.synchronize()
        print(f"n = {n}, {name:11}: {time.perf_counter() - start:.6f} seconds")

print("Coalesced matches NumPy:", np.allclose(C_dev.copy_to_host(), A @ B, rtol=1e-4))

n = 2048, uncoalesced: 0.154609 seconds
n = 2048, coalesced  : 0.081191 seconds
n = 4096, uncoalesced: 1.272617 seconds
n = 4096, coalesced  : 0.631783 seconds
Coalesced matches NumPy: True
